# Tool Pattern

<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f1a9fbda7-77a8-4a7a-ac2c-077fb98e53a6_716x552-1.gif" alt="Alt text" width="800"/>

---

As you may already know, the information stored in LLM weights is (usually) 𝐧𝐨𝐭 𝐞𝐧𝐨𝐮𝐠𝐡 to give accurate and insightful answers to our questions.
 
That's why we need to provide the LLM with ways to access the outside world. 🌍 

In practice, you can build tools for whatever you want (at the end of the day they are just functions the LLM can use), from a tool that let's you access Wikipedia, another to analyse the content of YouTube videos or calculate difficult integrals using Wolfram Alpha. 

The second pattern we are going to implement is the **tool pattern**. 

In this notebook, you'll learn how **tools** actually work. This is the **second lesson** of the "Agentic Patterns from Scratch" series. Take a look at the first lesson if you haven't!

* [First Lesson: The Reflection Pattern](https://github.com/neural-maze/agentic_patterns/blob/main/notebooks/reflection_pattern.ipynb)

## A simple function

Take a look at this function 👇

In [1]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Madrid, Barcelona
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Madrid":
		return json.dumps({"temperature": 25, "unit": unit})

	else:
		return json.dumps({"temperature": 58, "unit": unit})

Very simple, right? You provide a `location` and a `unit` and it returns the temperature.

In [2]:
get_current_weather(location="Madrid", unit="celsius")

'{"temperature": 25, "unit": "celsius"}'

But the question is:

**How can you make this function available to an LLM?**

An LLM is a type of NLP system, so it expects text as input. But how can we transform this function into text?

## A System Prompt that works

For the LLM to be aware of this function, we need to provide some relevant information about it in the context. **I'm referring to the function name, attributes, description, etc.** Take a look at the following System Prompt.

```xml
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "string"
            },
            "unit": {
                "type": "string"
            }
        }
    }
}
</tools>
```


As you can see, the LLM enforces the LLM to behave as a `function calling AI model` who, given a list of function signatures inside the <tools></tools> XML tags
will select which one to use. When the model decides a function to use, it will return a json like the following, representing a function call:

```xml
<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>
```


Let's see how it works in practise! 👇

In [3]:
import os
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root_candidates = [cwd, cwd.parent]
for candidate in repo_root_candidates:
    if (candidate / ".env.example").exists() and (candidate / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repo root")

env_path = repo_root / ".env"
if not env_path.exists():
    raise FileNotFoundError("Expected .env at the repo root. Copy .env.example to .env first.")

# Load the shared repo configuration so the notebook behaves the same from the lab folder or repo root.
load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from .env")
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query.
Do not make assumptions about argument values. If required arguments are missing, ask the user.

For each function call, return a JSON object with function name and arguments
inside <tool_call></tool_call> XML tags in the following format:

<tool_call>
{"name": "<function-name>", "arguments": {<args-dict>}}
</tool_call>

Here are the available tools:

<tools>
[
  {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "City and state, e.g., 'Madrid, Spain' or 'Baltimore, MD'"
        },
        "unit": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"],
          "description": "Temperature unit"
        }
      },
      "required": ["location"]
    }
  }
]
</tools>
"""

Let's ask a very simple question: `"What's the current temperature in Madrid, in Celsius?"`

In [4]:
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Madrid, in Celsius?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = client.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather", "arguments": {"location": "Madrid, Spain", "unit": "celsius"}}
</tool_call>


---

**That's an improvement!** We may not have the *proper* answer but, with this information, we can obtain it! How? Well, we just need to:

1. Parse the LLM output. By this I mean deleting the XML tags
2. Load the output as a proper Python dict

The function below does exactly this.

---

### Parse the Tool-Call Text

This helper function removes the wrapper tags and turns the model's tool-call text into Python data that the notebook can inspect and use.


In [5]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

### Inspect the Parsed Tool Call

Run this next cell to see the tool name and arguments after the XML-style wrapper has been stripped away. This helps you connect the model's text output to an actual Python function call.


In [6]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Madrid, Spain', 'unit': 'celsius'}}

We can simply run the function now, by passing the arguments like this 👇

In [7]:
tool_name = parsed_output["name"]
tool_args = parsed_output.get("arguments", {})

if tool_name == "get_current_weather":
    result = get_current_weather(**tool_args)
else:
    raise ValueError(f"Unknown tool: {tool_name}")

Display the raw tool result before you add it back into the conversation. This is the fact the model will use in its final answer.


In [8]:
result

'{"temperature": 58, "unit": "celsius"}'

**That's it!** A temperature of 25 degrees Celsius. 

As you can see, we're dealing with a string, so we can simply add the parsed_output to the `chat_history` so that the LLM knows the information it has to return to the user. 

Now that we have the raw tool result, add it back to the conversation as an `Observation` and ask the model for a final answer grounded in that tool output.


In [9]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

Now ask the model for its final answer using the tool result you just added as an observation.


In [10]:
client.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Madrid is **58°C**.'

## Implementing everything the good way

To recap, we have a way for the LLM to generate `tool_calls` that we can use later to *properly* run the functions. But, as you may imagine, there are some pieces missing:

1. We need to automatically transform any function into a description like we saw in the initial system prompt.
2. We need a way to tell the agent that this function is a tool

Let's do it!

### The `tool` decorator

We are going to use the `tool` decorator to transform any Python function into a tool. You can see the implementation [here](https://github.com/neural-maze/agentic_patterns/blob/main/src/agentic_patterns/tool_pattern/tool.py). To test it out, let's make a more complex tool than before. For example, a tool that interacts with [Hacker News](https://news.ycombinator.com/), getting the current top stories. 

> Reminder: To automatically generate the function signature for the tool, we need a way to infer the arguments types. For this reason, we need to create the typing annotations. 

In [11]:
import sys
from pathlib import Path

src_dir_candidates = [Path("src"), Path("../src")]
for candidate in src_dir_candidates:
    if (candidate / "agentic_patterns").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError("Could not find local src/agentic_patterns folder")

import json
import requests
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.tool_pattern.tool_agent import ToolAgent

def fetch_top_hacker_news_stories(top_n: int):
    """
    Fetch the top stories from Hacker News.

    This function retrieves the top `top_n` stories from Hacker News using the Hacker News API. 
    Each story contains the title, URL, score, author, and time of submission. The data is fetched 
    from the official Firebase Hacker News API, which returns story details in JSON format.

    Args:
        top_n (int): The number of top stories to retrieve.
    """
    top_stories_url = 'https://hacker-news.firebaseio.com/v0/topstories.json'
    
    try:
        response = requests.get(top_stories_url)
        response.raise_for_status()  # Check for HTTP errors
        
        # Get the top story IDs
        top_story_ids = response.json()[:top_n]
        
        top_stories = []
        
        # For each story ID, fetch the story details
        for story_id in top_story_ids:
            story_url = f'https://hacker-news.firebaseio.com/v0/item/{story_id}.json'
            story_response = requests.get(story_url)
            story_response.raise_for_status()  # Check for HTTP errors
            story_data = story_response.json()
            
            # Append the story title and URL (or other relevant info) to the list
            top_stories.append({
                'title': story_data.get('title', 'No title'),
                'url': story_data.get('url', 'No URL available'),
            })
        
        return json.dumps(top_stories)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return []

If we run this Python function, we'll obtain the top HN stories, as you can see below (the top 5 in this case).

In [12]:
json.loads(fetch_top_hacker_news_stories(top_n=5))

[{'title': 'MacBook Pro with M5 Pro and M5 Max',
  'url': 'https://www.apple.com/newsroom/2026/03/apple-introduces-macbook-pro-with-all-new-m5-pro-and-m5-max/'},
 {'title': "Lenovo's New ThinkPads Score 10/10 for Repairability",
  'url': 'https://www.ifixit.com/News/115827/new-thinkpads-score-perfect-10-repairability'},
 {'title': 'Motorola GrapheneOS devices will be bootloader unlockable/relockable',
  'url': 'https://grapheneos.social/@GrapheneOS/116160393783585567'},
 {'title': "Claude's Cycles [pdf]",
  'url': 'https://www-cs-faculty.stanford.edu/~knuth/papers/claude-cycles.pdf'},
 {'title': 'Voxile: A ray-traced game made in its own engine and programming language',
  'url': 'https://elbowgreasegames.substack.com/p/voxray-games-pushes-major-update'}]

To transform the `fetch_top_hacker_news_stories` function into a Tool, we can use the `tool` decorator.

In [13]:
hn_tool = tool(fetch_top_hacker_news_stories)

The Tool has the following parameters: a `name`, a `fn_signature` and the `fn` (this is the function we are going to call, this case `fetch_top_hacker_news_stories`)

In [14]:
hn_tool.name

'fetch_top_hacker_news_stories'

By default, the tool gets its name from the function name.

In [15]:
json.loads(hn_tool.fn_signature)

{'name': 'fetch_top_hacker_news_stories',
 'description': '\n    Fetch the top stories from Hacker News.\n\n    This function retrieves the top `top_n` stories from Hacker News using the Hacker News API. \n    Each story contains the title, URL, score, author, and time of submission. The data is fetched \n    from the official Firebase Hacker News API, which returns story details in JSON format.\n\n    Args:\n        top_n (int): The number of top stories to retrieve.\n    ',
 'parameters': {'properties': {'top_n': {'type': 'int'}}}}

As you can see, the function signature has been automatically generated. It contains the `name`, a `description` (taken from the docstrings) and the `parameters`, whose types come from the tying annotations. Now that we have a tool, let's run the agent.

### The `ToolAgent`

To create the agent, we just need to pass a list of tools (in this case, just one).

In [16]:
tool_agent = ToolAgent(tools=[hn_tool], client=client, model=MODEL)

A quick check to see that everything works fine. If we ask the agent something unrelated to Hacker News, it shouldn't use the tool.

In [17]:
output = tool_agent.run(user_msg="Tell me your name")

Display the agent response so you can check whether it stayed grounded in the tool output.


In [18]:
print(output)

Hello! My name is Qwen, and I'm a large language model developed by Alibaba Cloud. I'm here to help you with a wide range of tasks, such as answering questions, creating content, solving problems, and more. How can I assist you today? 😊


Now, let's ask for specific information about Hacker News.

In [19]:
output = tool_agent.run(user_msg="Tell me the top 5 Hacker News stories right now")


Using Tool: fetch_top_hacker_news_stories

Tool call dict: 
{'name': 'fetch_top_hacker_news_stories', 'arguments': {'top_n': 5}, 'id': 1}

Tool result: 
[{"title": "MacBook Pro with M5 Pro and M5 Max", "url": "https://www.apple.com/newsroom/2026/03/apple-introduces-macbook-pro-with-all-new-m5-pro-and-m5-max/"}, {"title": "Lenovo's New ThinkPads Score 10/10 for Repairability", "url": "https://www.ifixit.com/News/115827/new-thinkpads-score-perfect-10-repairability"}, {"title": "Motorola GrapheneOS devices will be bootloader unlockable/relockable", "url": "https://grapheneos.social/@GrapheneOS/116160393783585567"}, {"title": "Claude's Cycles [pdf]", "url": "https://www-cs-faculty.stanford.edu/~knuth/papers/claude-cycles.pdf"}, {"title": "Voxile: A ray-traced game made in its own engine and programming language", "url": "https://elbowgreasegames.substack.com/p/voxray-games-pushes-major-update"}]


Display the final Hacker News answer so you can inspect how the tool-enabled agent turns fetched data into a user-facing response.


In [20]:
print(output)

Here are the top 5 Hacker News stories based on the provided observation:

1. **[MacBook Pro with M5 Pro and M5 Max](https://www.apple.com/newsroom/2026/03/apple-introduces-macbook-pro-with-all-new-m5-pro-and-m5-max/)**  
   Apple unveils new MacBook Pro models with M5 Pro and M5 Max chips.

2. **[Lenovo's New ThinkPads Score 10/10 for Repairability](https://www.ifixit.com/News/115827/new-thinkpads-score-perfect-10-repairability)**  
   New ThinkPads earn top marks for ease of repair and modifiability.

3. **[Motorola GrapheneOS devices will be bootloader unlockable/relockable](https://grapheneos.social/@GrapheneOS/116160393783585567)**  
   GrapheneOS devices now support bootloader unlock and relock features.

4. **[Claude's Cycles [pdf]](https://www-cs-faculty.stanford.edu/~knuth/papers/claude-cycles.pdf)**  
   A research paper exploring Claude's approach to cycle detection and optimization.

5. **[Voxile: A ray-traced game made in its own engine and programming language](https://el

---
There you have it!! A fully functional Tool!! 🛠️